# Анализ данных — занятие 5
### Методы кластеризации: метрические, модельные и иерархические

**Логика занятия:** три семейства методов кластеризации → почему метрические методы ломаются на шуме → модельные методы и EM-алгоритм (Gaussian Mixture) → выбор числа компонент по BIC → иерархические методы → гибрид GMSDB → практика на словах.

## 1. Три семейства методов кластеризации

Все методы кластеризации делятся на три семейства:

- **Метрические** — работают с расстоянием между точками. Координаты не обязательны, достаточно знать расстояния. Гиперпараметр обычно связан с расстоянием (`eps`). Пример — `DBSCAN`.
- **Модельные** — предполагают, что точки порождены распределениями-моделями (например, гауссовыми «горбами»). Гиперпараметр — число моделей в смеси. Пример — `GaussianMixture`.
- **Иерархические** — последовательно делят или объединяют кластеры. Пример — `AgglomerativeClustering`.

Главное правило: у **любой** кластеризации есть хотя бы один гиперпараметр, и он всегда влияет на результат.

In [ ]:
import sys
!{sys.executable} -m pip install gmsdb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import numpy as np
from gmsdb import GMSDB

## 2. Метрические методы и шум

Возьмём `make_blobs` — три хорошо разделённых облака. `DBSCAN` с подбором `eps` по силуэту разделит их отлично.

In [ ]:
sopt, epsopt, copt = -1, -1, None
x, y = make_blobs(n_samples=1000, centers=3, n_features=2)

for eps in np.linspace(0.1, 1.5, 20):
    clf = DBSCAN(eps=eps)
    c = clf.fit_predict(x)
    if len(set(c)) > 1:
        s = silhouette_score(x, c)
        if s > sopt:
            sopt, epsopt, copt = s, eps, c

print('eps:', round(epsopt, 3), '| silhouette:', round(sopt, 4))
plt.scatter(x[:, 0], x[:, 1], c=copt)
plt.title('make_blobs: DBSCAN')
plt.show()

На датасете `make_moons` («две подковы») всё сложнее: кластеры не круглые. `noise` задаёт ширину подков.

In [ ]:
sopt, epsopt, copt = -1, -1, None
x, y = make_moons(n_samples=1000, noise=0.1)

for eps in np.linspace(0.1, 1.5, 20):
    clf = DBSCAN(eps=eps)
    c = clf.fit_predict(x)
    if len(set(c)) > 1:
        s = silhouette_score(x, c)
        if s > sopt:
            sopt, epsopt, copt = s, eps, c

print('eps:', round(epsopt, 3), '| silhouette:', round(sopt, 4))
plt.scatter(x[:, 0], x[:, 1], c=copt)
plt.title('make_moons: DBSCAN')
plt.show()

## 3. Почему метрические методы боятся шума

Добавим к хорошо разделённым облакам тысячу случайных точек-шумов. Формально кластеры те же, но `DBSCAN` перестаёт работать: метрический метод неустойчив к фоновому шуму. Он хорош только на хорошо разделённых кластерах без фона.

In [ ]:
sopt, epsopt, copt = -100, -1, None
X, y = make_blobs(n_samples=1000, n_features=2, centers=3, cluster_std=1)
X = np.concatenate((X, (np.random.rand(1000, 2) - 0.5) * 30))

for eps in np.linspace(0.1, 10, 100):
    clf = DBSCAN(eps=eps)
    c = clf.fit_predict(X)
    if len(set(c)) > 1:
        s = silhouette_score(X, c)
        if s > sopt:
            sopt, epsopt, copt = s, eps, c

plt.scatter(X[:, 0], X[:, 1], s=3, c=copt)
plt.title('DBSCAN на данных с шумом')
plt.show()

## 4. Модельные методы: Gaussian Mixture

Если кластеры не шарообразные или есть шум, нужны **модельные методы**. Они предполагают, что точки распределены по смеси распределений (обычно гауссовых), и устойчивы к шуму.

Сгущения точек представляют как «горбы» плотности вероятности. Задача — подобрать параметры горбов: амплитуды, центры `xᵢ` и ширины `σᵢ`. В многомерном случае расстояние до центра считается по Махаланобису (с учётом обратной ковариационной матрицы).

Для подбора параметров служит **EM-алгоритм** (`Expectation–Maximization`):

1. случайно раскидываем центры;
2. **E-шаг**: по текущим параметрам считаем, к какому распределению вероятнее относится каждая точка;
3. **M-шаг**: пересчитываем параметры (центры, ширины, амплитуды) по этим точкам;
4. повторяем, пока распределение не сойдётся.

Число компонент `n_components` — гиперпараметр.

In [ ]:
sopt, nopt, copt = np.inf, None, None
x, y = make_blobs(n_samples=1000, centers=3, n_features=2, cluster_std=1, random_state=42)
x = np.concatenate((x, (np.random.rand(1000, 2) - 0.5) * 30))

for n in range(2, 30):
    clf = GaussianMixture(n_components=n)
    c = clf.fit_predict(x)
    s = clf.bic(x)
    if s < sopt:
        sopt, nopt, copt = s, n, c

print('оптимальное число компонент:', nopt)
print('BIC:', round(sopt, 1))
plt.scatter(x[:, 0], x[:, 1], c=copt, s=3)
plt.title(f'GaussianMixture, n = {nopt}')
plt.show()

## 5. Как выбрать число компонент: BIC

Можно увеличивать число гауссов, пока не опишем все точки идеально, — но это бессмысленное переобучение.

Работают с **правдоподобием** — вероятностью того, что все точки принадлежат модели. Так как вероятности меньше единицы, их перемножают через **сумму логарифмов** и получают логарифмическое правдоподобие.

Далее действует **бритва Оккама** — «не множить сущности». К правдоподобию добавляют штраф, линейно растущий с числом параметров. Получается **байесовский информационный критерий (BIC)**: с ростом числа гауссов он сначала падает, затем штраф перевешивает, и появляется минимум. Минимум BIC соответствует оптимальному числу компонент.

Значение BIC не интерпретируется — важен только его минимум.

## 6. Иерархические методы

Недостаток метрических и модельных методов — резкая зависимость от гиперпараметра: чуть сдвинули `eps` — результат развалился. **Иерархические методы** устойчивее: небольшое изменение гиперпараметра чуть-чуть меняет результат.

- **Дивизивный** метод: сначала все точки в одном кластере, затем самый большой кластер делится надвое, пока в каждом кластере не останется по точке. Гиперпараметр — итоговое число кластеров.
- **Агломеративный** метод: сначала каждая точка — свой кластер, затем два самых близких кластера объединяются. Проще алгоритмизуется, поэтому используется чаще.

In [ ]:
X, y = make_blobs(n_samples=1000, n_features=2, centers=3, cluster_std=1, random_state=42)
X = np.concatenate((X, (np.random.rand(1000, 2) - 0.5) * 30))

sopt, nopt, copt = -1, None, None
for n in range(2, 30):
    clf = AgglomerativeClustering(n_clusters=n)
    c = clf.fit_predict(X)
    if len(set(c)) > 1:
        s = silhouette_score(X, c)
        if s > sopt:
            sopt, nopt, copt = s, n, c

print('оптимальное число кластеров:', nopt, '| silhouette:', round(sopt, 4))
plt.scatter(X[:, 0], X[:, 1], s=3, c=copt)
plt.title('AgglomerativeClustering')
plt.show()

## 7. GMSDB: GMM + DBSCAN с уровнем значимости

Можно построить метод, у которого гиперпараметр почти всегда имеет одно понятное значение — **уровень значимости** (`α ≈ 0.05`).

Идея `GMSDB`: сначала разделить данные через `GaussianMixture`, а затем объединить близкие гауссовы компоненты по принципу `DBSCAN`, используя расстояние Махаланобиса (фактически Z-статистику). Платят за это временем вычислений.

Единственный параметр — верхняя граница числа компонент (`n_components`), которую берут с запасом.

In [ ]:
X, y = make_blobs(n_samples=1000, n_features=2, centers=3, cluster_std=1, random_state=42)
X = np.concatenate((X, (np.random.rand(1000, 2) - 0.5) * 30))

clf = GMSDB(n_components=25)
c = clf.fit_predict(X)
plt.scatter(X[:, 0], X[:, 1], s=3, c=c)
plt.title('GMSDB на blobs с шумом')
plt.show()

Проверим `GMSDB` на сложном датасете `make_moons`, где обычный `GaussianMixture` не справлялся:

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = np.concatenate((X, (np.random.rand(1000, 2) - 0.5) * 5))

clf = GMSDB(n_components=25)
c = clf.fit_predict(X)
plt.scatter(X[:, 0], X[:, 1], s=3, c=c)
plt.title('GMSDB на make_moons')
plt.show()

## 8. Практика: кластеризация слов

Кластеризуем слова из `files/words.txt`. Самый простой признак — длина слова. Построим матрицу, где для каждого слова хранится его длина, и кластеризуем её `AgglomerativeClustering`.

In [ ]:
f = open('files/words.txt', encoding='utf-8')
d = []
for s in f:
    s = s.replace('\n', '')
    d.append([s, len(s)])
f.close()
d = np.array(d)

X = d[:, 1].astype(float)[:, np.newaxis]

for n in range(2, 15):
    clf = AgglomerativeClustering(n_clusters=n)
    c = clf.fit_predict(X)
    if len(set(c)) > 1:
        s = silhouette_score(X, c)
        print(f'n = {n}: silhouette = {s:.4f}')

---

**Итог занятия 5.** Разобрали три семейства методов кластеризации. Метрические методы (`DBSCAN`) неустойчивы к шуму, модельные (`GaussianMixture`) к нему устойчивы, но ограничены заложенной моделью, а число компонент подбирается по BIC. Разобрали иерархические методы (дивизивный и агломеративный) и гибрид `GMSDB`, у которого гиперпараметр — уровень значимости.